In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tabulate import tabulate
import warnings
from sklearn_genetic import GASearchCV
from sklearn_genetic.space import Categorical, Integer, Continuous
from sklearn.metrics import make_scorer
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split,  cross_validate
from sklearn.preprocessing import StandardScaler
import time
import sys, os
from sklearn.pipeline import make_pipeline
from skopt import BayesSearchCV
import skopt.space.space
from sklearn.model_selection import KFold


path = 'datasets/Dataset_2.xlsx'

# Read and load dataset
df= pd.read_excel(path, sheet_name="preproc")





/home/demetra/Desktop/Demetra/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INPUT DATA

In [ ]:
X = (df.iloc[:,14:]).values
print(X.shape)
std_scaler = StandardScaler()

NameError: name 'df' is not defined

OUTPUT DATA

In [3]:
Y = (df.iloc[:,7]).values
print(Y.shape)

(2905,)


**NRMSEPiqr**

In [4]:


def NRMSEPiqr(observed_values, predicted_values):
    # Calculate RMSEP
    rmsep = np.sqrt(np.mean((observed_values - predicted_values) ** 2))
    # Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1 = np.percentile(observed_values, 25)
    Q3 = np.percentile(observed_values, 75)

    # Calculate IQR
    IQR = Q3 - Q1

    return rmsep/IQR

NRMSEPiqrscorer = make_scorer(NRMSEPiqr,greater_is_better=False)


SVR

In [5]:
if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore" 


trials = 100
ranndom_cv = np.random.seed(777)
seeds = np.random.choice(trials, size=trials, replace= False)

ripartitions = [0.25, 0.50, 0.75]

results = []  

for ripartition in ripartitions:
        
    run_name = str(ripartition) + " ripartition"
    print(run_name)
    for i in range(trials):
        result = {"Ripartition":None,
                   "Trial":None,
                    "C_bayesian":None,
                    # "C_evolutionary":None,
                      "epsilon_bayesian":None,
                     #  "epsilon_evolutionary":None,
                        "bayesian_time":None,
                      #    "evolutionary_time" :None,
                            "NRMSEPiqr_bayesian":None,
                       #       "NRMSEPiqr_evolutionary":None,
                                "seed": None}
            

        print("Trial:",i+1)
                
        result["Ripartition"] = ripartition
        result["Trial"] = i +1  
        result["seed"] = seeds[i]
        X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=ripartition, random_state=seeds[i])
        '''     
        param_grid = {
            'C': Continuous(1/10000,10),
            "epsilon":Continuous(1/100000,1),
            "gamma": 
            
            }
          '''
        svr_regressor =  SVR(kernel="linear", gamma="scale")
        
        kf = KFold(n_splits=5, shuffle=True,random_state=ranndom_cv)
        #sturges = int(1 + np.log(len(X_train)))    
        '''           
        evolved_estimator = GASearchCV(estimator=svr_regressor,
                                cv=kf,
                                scoring=NRMSEPiqrscorer,
                                param_grid=param_grid,
                                n_jobs=-1,
                                verbose=False,
                                population_size=10,
                                generations=10)  
        
        '''
        bayesian_estimator = BayesSearchCV(
                                svr_regressor,
                                {
                                'C':skopt.space.space.Integer(1,5000,"log-uniform"),
                                "epsilon":skopt.space.space.Real(1e-4,1,"log-uniform")},
                                cv=kf,
                                scoring=NRMSEPiqrscorer,
                                n_jobs=-1,
                                verbose= False,
                                 n_iter=15 )
            
        '''
        start_time = time.time()
        evolved_estimator.fit(X_train, Y_train)
        elapsed_time = (time.time() -start_time) #elapsed time in seconds

        result["evolutionary_time"] = elapsed_time
        result["C_evolutionary"] = float(evolved_estimator.best_params_["C"])
        result["epsilon_evolutionary"] = float(evolved_estimator.best_params_["epsilon"])
       '''
        start_time = time.time()
        bayesian_estimator.fit(X_train, Y_train)
        elapsed_time = (time.time() -start_time) #elapsed time in seconds

        result["bayesian_time"] = elapsed_time
        result["C_bayesian"] = float(bayesian_estimator.best_params_["C"])
        result["epsilon_bayesian"] = float(bayesian_estimator.best_params_["epsilon"])
           
        #result["NRMSEPiqr_evolutionary"] = NRMSEPiqr(Y_test, evolved_estimator.predict(X_test))
    

        result["NRMSEPiqr_bayesian"] =  NRMSEPiqr(Y_test,bayesian_estimator.predict(X_test))
    
        print(result)
        results.append(result)

0.25 ripartition
Trial: 1
{'Ripartition': 0.25, 'Trial': 1, 'C_bayesian': 4949.0, 'epsilon_bayesian': 0.14179505780012844, 'bayesian_time': 92.05855393409729, 'NRMSEPiqr_bayesian': 0.31809139814871035, 'seed': 26}
Trial: 2
{'Ripartition': 0.25, 'Trial': 2, 'C_bayesian': 2905.0, 'epsilon_bayesian': 0.001235229772716577, 'bayesian_time': 120.15870881080627, 'NRMSEPiqr_bayesian': 0.33910964295342866, 'seed': 75}
Trial: 3
{'Ripartition': 0.25, 'Trial': 3, 'C_bayesian': 2308.0, 'epsilon_bayesian': 0.0001, 'bayesian_time': 65.54435229301453, 'NRMSEPiqr_bayesian': 0.328489755592321, 'seed': 90}
Trial: 4
{'Ripartition': 0.25, 'Trial': 4, 'C_bayesian': 4508.0, 'epsilon_bayesian': 0.0011584533018218757, 'bayesian_time': 96.16589546203613, 'NRMSEPiqr_bayesian': 0.32175200897314676, 'seed': 96}
Trial: 5
{'Ripartition': 0.25, 'Trial': 5, 'C_bayesian': 2619.0, 'epsilon_bayesian': 0.005178711953948926, 'bayesian_time': 97.13505959510803, 'NRMSEPiqr_bayesian': 0.321256722946611, 'seed': 1}
Trial: 6
{'

In [6]:


df = pd.DataFrame(results)
df.to_excel("SVR_Bayes.xlsx")


In [7]:
from collections import Counter


svr_C = {ripartitions[0]:0,ripartitions[1]:0,ripartitions[2]:0}
svr_epsilon= {ripartitions[0]:0,ripartitions[1]:0,ripartitions[2]:0}
svr_kernel = {ripartitions[0]:None,ripartitions[1]:None,ripartitions[2]:None}
svr_gamma= {ripartitions[0]:None,ripartitions[1]:None,ripartitions[2]:None}

#Compute the mean for every ripartition
for trial in results:
    svr_C[trial["Ripartition"]]+=trial["C_bayesian"]/trials
    svr_epsilon[trial["Ripartition"]]+=trial["epsilon_bayesian"]/trials


print("Average C: ",svr_C)
print("Average epsilon: ",svr_epsilon)

Average C:  {0.25: 3660.2099999999996, 0.5: 3908.95, 0.75: 3939.1799999999994}
Average epsilon:  {0.25: 0.021238631291644737, 0.5: 0.027625141089676886, 0.75: 0.016366943749828725}


In [ ]:

trials = 100
random_cv = np.random.seed(777)
seeds = np.random.choice(trials, size=trials, replace = False)
ripartitions = [0.25, 0.50, 0.75]
svr_C = {ripartitions[0]:5894,ripartitions[1]:6130,ripartitions[2]:5995}
svr_epsilon= {ripartitions[0]:0.0166776518250441,ripartitions[1]:0.0184257995255977,ripartitions[2]:0.0248397548439159}
svr_gamma= {ripartitions[0]:0.000176103767093283,ripartitions[1]:0.000171615010508122,ripartitions[2]:0.000213372354004179}


final_results_svr = []
for C, epsilon,gamma,ripartition in zip (svr_C.values(), svr_epsilon.values(),svr_gamma.values(),ripartitions):
    predictions_svr = []
    result_NRMSEP_svr = []
    NRMSE = []
    total_time = 0
    for i in range(trials):
        X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=ripartition, random_state=seeds[i])
        svr_regressor = make_pipeline(StandardScaler(),SVR(C=C,epsilon=epsilon, kernel="rbf", gamma=gamma))
            
        start_time = time.time()
        svr_regressor.fit(X_train, Y_train)
        elapsed_time = (time.time() -start_time) #elapsed time in seconds
        
        # Make predictions using the testing set
        Y_pred_svr = svr_regressor.predict(X_test)

        
        NRMSE.append(NRMSEPiqr(Y_test, Y_pred_svr))
        total_time+=elapsed_time
        

        for k in range(len(Y_pred_svr)):

          predictions_svr.append({"Ripartition":str(ripartition *100 ),
                        "NRMSEPiqr": NRMSEPiqr(Y_test, Y_pred_svr),
                        "Training time": elapsed_time, 
                        "C": C, 
                        "epsilon": epsilon,
                        "Seed": seeds[i],
                        "Prediction": Y_pred_svr[k], 
                        "Actual": Y_test[k] })
        
        result_NRMSEP_svr.append({"Ripartition":str(ripartition *100 ),
                        "NRMSEPiqr": NRMSEPiqr(Y_test, Y_pred_svr),
                        "Training time": elapsed_time, 
                        "C": C, 
                        "epsilon": epsilon,
                        "Seed": seeds[i],
        })
    final_results_svr.append({"Test size(%)": str(ripartition *100 ),
                          "Mean NRMSEPiqr":np.mean(NRMSE), 
                          "Var NRMSEPiqr": np.var(NRMSE),
                          "Mean Training Time (sec)":total_time/trials, 
                          "C": C, 
                          "epsilon": epsilon,})
    
    df = pd.DataFrame(predictions_svr)
    df.to_excel(f"SVR_{ripartition}_predictions.xlsx")
    df = pd.DataFrame(result_NRMSEP_svr)
    df.to_excel(f"SVR_{ripartition}_NRMSEP.xlsx")
df = pd.DataFrame(final_results_svr)
df.to_excel(f"SVR_final.xlsx")